# Sparse Probing

L1-regularized probes for finding minimal feature sets that encode
concepts: sparse directions, feature selection, minimal probe sets,
and sparse vs dense comparison.

Reference: Gurnee et al. (2023) "Finding Neurons in a Haystack"

## Why This Matters

Sparse probing investigates representations that are sparse — where only a few components are active for any given input. Sparsity is a key property of interpretable representations, as it enables individual features to be studied in isolation.

**Key references:**
- [Bricken et al. (2023) "Towards Monosemanticity"](https://transformer-circuits.pub/2023/monosemantic-features/index.html) — Sparse autoencoders find interpretable features

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.sparse_probing import (
    sparse_linear_probe,
    sparse_concept_direction,
    feature_selection_probe,
    minimal_probe_set,
    sparse_vs_dense_comparison,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens_list = [
    jnp.array([1, 2, 3, 4, 5, 6, 7]),
    jnp.array([2, 3, 4, 5, 6, 7, 8]),
    jnp.array([50, 51, 52, 53, 54, 55, 56]),
    jnp.array([51, 52, 53, 54, 55, 56, 57]),
    jnp.array([3, 4, 5, 6, 7, 8, 9]),
    jnp.array([52, 53, 54, 55, 56, 57, 58]),
]
labels = [0, 0, 1, 1, 0, 1]
print("Model and data ready")

## Sparse Linear Probe

In [ ]:
result = sparse_linear_probe(model, tokens_list, labels, l1_strength=0.1)
print(f"Accuracy: {result['accuracy']:.1%}")
print(f"Sparsity: {result['sparsity']:.1%}")
print(f"Active dimensions ({result['n_active']}): {result['active_dimensions']}")

## Sparse Concept Direction

In [ ]:
result = sparse_concept_direction(model, tokens_list, labels, max_dims=5)
print(f"Selected dims: {result['selected_dims']}")
print(f"Accuracy curve: {[f'{a:.1%}' for a in result['accuracy_curve']]}")
print(f"Separation: {result['separation']:.4f}")

## Feature Selection Probe

In [ ]:
result = feature_selection_probe(model, tokens_list, labels, max_features=8)
print(f"Selected features: {result['selected_features']}")
print(f"Accuracy per step: {[f'{a:.1%}' for a in result['accuracy_per_feature']]}")
print(f"Final accuracy: {result['final_accuracy']:.1%}")

## Minimal Probe Set

In [ ]:
result = minimal_probe_set(model, tokens_list, labels, target_accuracy=0.8)
print(f"Dims needed: {result['n_dims_needed']}")
print(f"Accuracy reached: {result['accuracy_at_threshold']:.1%}")
print(f"Full accuracy: {result['full_accuracy']:.1%}")

## Sparse vs Dense Comparison

In [ ]:
result = sparse_vs_dense_comparison(model, tokens_list, labels, n_sparse_dims=5)
print(f"Sparse accuracy ({result['n_sparse_dims']} dims): {result['sparse_accuracy']:.1%}")
print(f"Dense accuracy ({result['n_dense_dims']} dims): {result['dense_accuracy']:.1%}")
print(f"Gap: {result['gap']:+.1%}")
print(f"Efficiency ratio: {result['efficiency_ratio']:.2f}")